In [112]:
# ============================================================
# Imports
# ============================================================

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from torch.utils.data import (
    Dataset,
    DataLoader
)

In [113]:
# ============================================================
# Load data
# ============================================================

X = np.load(
    r"C:/Users/Ander/Desktop/F1-project/data/processed/all_race_laps_tensor_clean.npy"
)

laps = pd.read_parquet(
    r"C:/Users/Ander/Desktop/F1-project/data/processed/all_race_laps_metadata_enriched.parquet"
)

# ============================================================
# Remove abnormal laps
# ============================================================

# ============================================================
# Filter laps first
# ============================================================

mask = (
    (laps["lap_time"] > 70) &
    (laps["lap_time"] < 115)
)

laps = laps[mask].reset_index(drop=True)

# apply SAME mask to telemetry tensor


print(X.shape)
print(laps.shape)

display(laps.tail())

(26647, 500, 10)
(24763, 18)


,idx,race,driver,lap,lap_time,compound,stint,life,fresh,wAT,wTT,wH,wP,wR,wWS,drv,team,sesT
24758,26661,United States Grand Prix,VER,52,98.192,SOFT,2.0,21.0,False,29.3,45.8,21.0,1000.6,False,1.4,VER,Red Bull Racing,8650.481
24759,26662,United States Grand Prix,VER,53,98.107,SOFT,2.0,22.0,False,29.1,45.7,21.0,1000.5,False,1.2,VER,Red Bull Racing,8748.588
24760,26663,United States Grand Prix,VER,54,98.899,SOFT,2.0,23.0,False,29.2,45.6,20.0,1000.6,False,1.7,VER,Red Bull Racing,8847.487
24761,26664,United States Grand Prix,VER,55,98.958,SOFT,2.0,24.0,False,29.2,45.3,20.0,1000.6,False,2.8,VER,Red Bull Racing,8946.445
24762,26665,United States Grand Prix,VER,56,98.260,SOFT,2.0,25.0,False,29.1,44.8,21.0,1000.5,False,3.3,VER,Red Bull Racing,9044.705


In [114]:
# ============================================================
# Raw lap time target
# ============================================================

y = laps["lap_time"].values.astype(
    np.float32
)

print(pd.Series(y).describe())

count    24763.000000
mean        88.952812
std         10.434524
min         70.012001
25%         80.739498
50%         89.064003
75%         97.459999
max        114.998001
dtype: float64


In [115]:
# ============================================================
# Select first quarter of telemetry
# ============================================================

X_tel = X[:, :125, :]

X_tel = X_tel[mask.values]

print(X_tel.shape)

(24763, 125, 10)


In [116]:
# ============================================================
# Context features
# ============================================================

context_cols = [
    "wAT",   # air temp
    "wTT",   # track temp
    "wH",    # humidity
    "wP",    # pressure
    "wR"     # rainfall
]

X_ctx = laps[
    context_cols
].values.astype(np.float32)

print(X_ctx[0])

[  26.8   31.4   60.  1016.5    0. ]


In [117]:
# ============================================================
# Encode categorical variables
# ============================================================

from sklearn.preprocessing import LabelEncoder

driver_encoder = LabelEncoder()
race_encoder = LabelEncoder()
compound_encoder = LabelEncoder()
team_encoder = LabelEncoder()

categorical_cols = [
    "driver",
    "race",
    "compound",
    "team"
]

for col in categorical_cols:

    laps[col] = (
        laps[col]
        .fillna("UNKNOWN")
        .astype(str)
    )

driver_ids = driver_encoder.fit_transform(
    laps["driver"]
)

race_ids = race_encoder.fit_transform(
    laps["race"]
)

compound_ids = compound_encoder.fit_transform(
    laps["compound"]
)

team_ids = team_encoder.fit_transform(
    laps["team"]
)

num_drivers = len(
    driver_encoder.classes_
)

num_races = len(
    race_encoder.classes_
)

num_compounds = len(
    compound_encoder.classes_
)

num_teams = len(
    team_encoder.classes_
)

print(f"Drivers: {num_drivers}")
print(f"Races: {num_races}")
print(f"Compounds: {num_compounds}")
print(f"Teams: {num_teams}")

Drivers: 21
Races: 24
Compounds: 5
Teams: 10


In [118]:
# ============================================================
# Normalize telemetry
# ============================================================

N, T, F = X_tel.shape

telemetry_scaler = StandardScaler()

X_tel_2d = X_tel.reshape(-1, F)

X_tel_scaled = telemetry_scaler.fit_transform(
    X_tel_2d
)

X_tel_scaled = X_tel_scaled.reshape(
    N,
    T,
    F
)

In [119]:
# ============================================================
# Normalize context
# ============================================================

context_scaler = StandardScaler()

X_ctx_scaled = context_scaler.fit_transform(
    X_ctx
)

In [120]:
# ============================================================
# Normalize target
# ============================================================

y = laps["lap_time"].values.astype(
    np.float32
)

y_mean = y.mean()
y_std = y.std()

y_scaled = (
    (y - y_mean)
    / y_std
)

In [121]:
# ============================================================
# Sequential split within each race/driver
# ============================================================

train_indices = []
test_indices = []

grouped = laps.groupby(
    ["race", "driver"]
)

for _, group in grouped:

    group = group.sort_values(
        "lap"
    )

    n = len(group)

    split_idx = int(n * 0.8)

    train_indices.extend(
        group.index[:split_idx]
    )

    test_indices.extend(
        group.index[split_idx:]
    )

train_idx = np.array(train_indices)
test_idx = np.array(test_indices)

# ============================================================
# Split telemetry
# ============================================================

X_tel_train = X_tel_scaled[
    train_idx
]

X_tel_test = X_tel_scaled[
    test_idx
]

# ============================================================
# Split context
# ============================================================

X_ctx_train = X_ctx_scaled[
    train_idx
]

X_ctx_test = X_ctx_scaled[
    test_idx
]

# ============================================================
# Split embeddings
# ============================================================

driver_train = driver_ids[
    train_idx
]

driver_test = driver_ids[
    test_idx
]

race_train = race_ids[
    train_idx
]

race_test = race_ids[
    test_idx
]

compound_train = compound_ids[
    train_idx
]

compound_test = compound_ids[
    test_idx
]

team_train = team_ids[
    train_idx
]

team_test = team_ids[
    test_idx
]

# ============================================================
# Split targets
# ============================================================

y_train = y_scaled[
    train_idx
]

y_test = y_scaled[
    test_idx
]

print(X_tel_train.shape)
print(X_tel_test.shape)

(19605, 125, 10)
(5158, 125, 10)


In [122]:
# ============================================================
# Dataset
# ============================================================

class LapDataset(Dataset):

    def __init__(
        self,
        X_tel,
        X_ctx,
        driver_ids,
        race_ids,
        compound_ids,
        team_ids,
        y
    ):

        self.X_tel = torch.tensor(
            X_tel,
            dtype=torch.float32
        )

        self.X_ctx = torch.tensor(
            X_ctx,
            dtype=torch.float32
        )

        self.driver_ids = torch.tensor(
            driver_ids,
            dtype=torch.long
        )

        self.race_ids = torch.tensor(
            race_ids,
            dtype=torch.long
        )

        self.compound_ids = torch.tensor(
            compound_ids,
            dtype=torch.long
        )

        self.team_ids = torch.tensor(
            team_ids,
            dtype=torch.long
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.y)

    def __getitem__(self, idx):

        return (
            self.X_tel[idx],
            self.X_ctx[idx],
            self.driver_ids[idx],
            self.race_ids[idx],
            self.compound_ids[idx],
            self.team_ids[idx],
            self.y[idx]
        )

In [123]:
train_dataset = LapDataset(
    X_tel_train,
    X_ctx_train,
    driver_train,
    race_train,
    compound_train,
    team_train,
    y_train
)

test_dataset = LapDataset(
    X_tel_test,
    X_ctx_test,
    driver_test,
    race_test,
    compound_test,
    team_test,
    y_test
)

In [124]:
# ============================================================
# Dataloaders
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

In [125]:
# ============================================================
# LSTM model
# ============================================================

class LapNet(nn.Module):

    def __init__(
        self,
        telemetry_features,
        context_features,
        num_drivers,
        num_races,
        num_compounds,
        num_teams
    ):

        super().__init__()

        # ----------------------------------------------------
        # telemetry branch
        # ----------------------------------------------------

        self.lstm = nn.LSTM(
            input_size=telemetry_features,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        # ----------------------------------------------------
        # context branch
        # ----------------------------------------------------

        self.context_mlp = nn.Sequential(

            nn.Linear(
                context_features,
                32
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                32,
                32
            ),

            nn.ReLU()
        )

        # ----------------------------------------------------
        # embeddings
        # ----------------------------------------------------

        self.driver_embedding = nn.Embedding(
            num_drivers,
            8
        )

        self.race_embedding = nn.Embedding(
            num_races,
            8
        )

        self.compound_embedding = nn.Embedding(
            num_compounds,
            4
        )

        self.team_embedding = nn.Embedding(
            num_teams,
            6
        )

        # ----------------------------------------------------
        # final head
        # ----------------------------------------------------

        self.head = nn.Sequential(

            nn.Linear(
                128 + 32 + 8 + 8 + 4 + 6,
                64
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                64,
                1
            )
        )

    def forward(
        self,
        telemetry,
        context,
        driver_id,
        race_id,
        compound_id,
        team_id
    ):

        # telemetry:
        # (batch, time, features)

        lstm_out, (hidden, cell) = self.lstm(
            telemetry
        )

        # final LSTM layer hidden state
        x_tel = hidden[-1]

        x_ctx = self.context_mlp(
            context
        )

        x_driver = self.driver_embedding(
            driver_id
        )

        x_race = self.race_embedding(
            race_id
        )

        x_compound = self.compound_embedding(
            compound_id
        )

        x_team = self.team_embedding(
            team_id
        )

        x = torch.cat([
            x_tel,
            x_ctx,
            x_driver,
            x_race,
            x_compound,
            x_team
        ], dim=1)

        return self.head(x).squeeze()

In [126]:
model = LapNet(
    telemetry_features=F,
    context_features=X_ctx.shape[1],
    num_drivers=num_drivers,
    num_races=num_races,
    num_compounds=num_compounds,
    num_teams=num_teams
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print(model)

LapNet(
  (lstm): LSTM(10, 128, num_layers=2, batch_first=True, dropout=0.3)
  (context_mlp): Sequential(
    (0): Linear(in_features=5, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=32, bias=True)
    (4): ReLU()
  )
  (driver_embedding): Embedding(21, 8)
  (race_embedding): Embedding(24, 8)
  (compound_embedding): Embedding(5, 4)
  (team_embedding): Embedding(10, 6)
  (head): Sequential(
    (0): Linear(in_features=186, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [127]:
# ============================================================
# Training loop
# ============================================================

epochs = 10

for epoch in range(epochs):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    train_loss = 0

    for (
        X_tel_batch,
        X_ctx_batch,
        driver_batch,
        race_batch,
        compound_batch,
        team_batch,
        y_batch
    ) in train_loader:

        optimizer.zero_grad()

        pred = model(
            X_tel_batch,
            X_ctx_batch,
            driver_batch,
            race_batch,
            compound_batch,
            team_batch
        )

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(
        train_loader
    )

    # --------------------------------------------------------
    # TEST
    # --------------------------------------------------------

    model.eval()

    test_loss = 0

    with torch.no_grad():

        for (
            X_tel_batch,
            X_ctx_batch,
            driver_batch,
            race_batch,
            compound_batch,
            team_batch,
            y_batch
        ) in test_loader:

            pred = model(
                X_tel_batch,
                X_ctx_batch,
                driver_batch,
                race_batch,
                compound_batch,
                team_batch
            )

            loss = criterion(
                pred,
                y_batch
            )

            test_loss += loss.item()

    test_loss /= len(
        test_loader
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train: {train_loss:.4f} | "
        f"Test: {test_loss:.4f}"
    )

Epoch 01 | Train: 0.3894 | Test: 0.1734
Epoch 02 | Train: 0.1573 | Test: 0.1548
Epoch 03 | Train: 0.1430 | Test: 0.1959
Epoch 04 | Train: 0.1494 | Test: 0.1576
Epoch 05 | Train: 0.1368 | Test: 0.1430
Epoch 06 | Train: 0.1287 | Test: 0.1421
Epoch 07 | Train: 0.1082 | Test: 0.1284
Epoch 08 | Train: 0.1017 | Test: 0.1172
Epoch 09 | Train: 0.0900 | Test: 0.1237
Epoch 10 | Train: 0.0860 | Test: 0.1220


In [128]:
# ============================================================
# Evaluate
# ============================================================

model.eval()

predictions = []
targets = []

with torch.no_grad():

    for (
        X_tel_batch,
        X_ctx_batch,
        driver_batch,
        race_batch,
        compound_batch,
        team_batch,
        y_batch
    ) in test_loader:

        pred = model(
            X_tel_batch,
            X_ctx_batch,
            driver_batch,
            race_batch,
            compound_batch,
            team_batch
        )

        predictions.append(
            pred.numpy()
        )

        targets.append(
            y_batch.numpy()
        )

predictions = np.concatenate(
    predictions
)

targets = np.concatenate(
    targets
)

# back to seconds
pred_seconds = (
    predictions * y_std
    + y_mean
)

target_seconds = (
    targets * y_std
    + y_mean
)

mae = np.mean(
    np.abs(
        pred_seconds
        - target_seconds
    )
)

print("MAE (seconds):", mae)

MAE (seconds): 2.050353


In [129]:
print(y_mean, y_std)

88.95281 10.434313


In [130]:
display(
    laps["lap_time"].describe()
)

count    24763.000000
mean        88.952804
std         10.434523
min         70.012000
25%         80.739500
50%         89.064000
75%         97.460000
max        114.998000
Name: lap_time, dtype: float64

In [132]:
# ============================================================
# Predict ONE real Leclerc Monaco lap
# ============================================================

model.eval()

# choose lap number
target_lap = 25

# find matching row
row_idx = laps[
    (laps["race"] == "Monaco Grand Prix") &
    (laps["driver"] == "LEC") &
    (laps["lap"] == target_lap)
].index[0]

# ============================================================
# Build tensors
# ============================================================

x_tel = torch.tensor(
    X_tel_scaled[row_idx],
    dtype=torch.float32
).unsqueeze(0)

x_ctx = torch.tensor(
    X_ctx_scaled[row_idx],
    dtype=torch.float32
).unsqueeze(0)

driver_tensor = torch.tensor(
    [driver_ids[row_idx]],
    dtype=torch.long
)

race_tensor = torch.tensor(
    [race_ids[row_idx]],
    dtype=torch.long
)

compound_tensor = torch.tensor(
    [compound_ids[row_idx]],
    dtype=torch.long
)

team_tensor = torch.tensor(
    [team_ids[row_idx]],
    dtype=torch.long
)

# ============================================================
# Predict
# ============================================================

with torch.no_grad():

    pred_scaled = model(
        x_tel,
        x_ctx,
        driver_tensor,
        race_tensor,
        compound_tensor,
        team_tensor
    )

pred_seconds = (
    pred_scaled.item() * y_std
    + y_mean
)

actual_seconds = laps.iloc[row_idx][
    "lap_time"
]

print(f"Lap: {target_lap}")
print(f"Actual:    {actual_seconds:.3f}")
print(f"Predicted: {pred_seconds:.3f}")
print(
    f"Error:     "
    f"{abs(pred_seconds - actual_seconds):.3f} sec"
)

Lap: 25
Actual:    74.618
Predicted: 77.100
Error:     2.482 sec


In [134]:
# ============================================================
# Save trained LSTM model
# ============================================================

torch.save(
    {
        "model_state_dict": model.state_dict(),

        "driver_encoder": driver_encoder,
        "race_encoder": race_encoder,
        "compound_encoder": compound_encoder,
        "team_encoder": team_encoder,

        "y_mean": y_mean,
        "y_std": y_std
    },

    r"C:/Users/Ander/Desktop/F1-project/trained_models/lstm_trained.pth"
)

print("Saved LSTM model.")

Saved LSTM model.


In [133]:
# ============================================================
# GRU model
# ============================================================

class LapNet(nn.Module):

    def __init__(
        self,
        telemetry_features,
        context_features,
        num_drivers,
        num_races,
        num_compounds,
        num_teams
    ):

        super().__init__()

        # ----------------------------------------------------
        # telemetry branch
        # ----------------------------------------------------

        self.gru = nn.GRU(
            input_size=telemetry_features,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.5
        )

        # ----------------------------------------------------
        # context branch
        # ----------------------------------------------------

        self.context_mlp = nn.Sequential(

            nn.Linear(
                context_features,
                32
            ),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(
                32,
                32
            ),

            nn.ReLU()
        )

        # ----------------------------------------------------
        # embeddings
        # ----------------------------------------------------

        self.driver_embedding = nn.Embedding(
            num_drivers,
            8
        )

        self.race_embedding = nn.Embedding(
            num_races,
            8
        )

        self.compound_embedding = nn.Embedding(
            num_compounds,
            4
        )

        self.team_embedding = nn.Embedding(
            num_teams,
            6
        )

        # ----------------------------------------------------
        # final head
        # ----------------------------------------------------

        self.head = nn.Sequential(

            nn.Linear(
                64 + 32 + 8 + 8 + 4 + 6,
                64
            ),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(
                64,
                1
            )
        )

    def forward(
        self,
        telemetry,
        context,
        driver_id,
        race_id,
        compound_id,
        team_id
    ):

        # ----------------------------------------------------
        # GRU
        # ----------------------------------------------------

        gru_out, hidden = self.gru(
            telemetry
        )

        # final hidden state
        x_tel = hidden[-1]

        # ----------------------------------------------------
        # context
        # ----------------------------------------------------

        x_ctx = self.context_mlp(
            context
        )

        # ----------------------------------------------------
        # embeddings
        # ----------------------------------------------------

        x_driver = self.driver_embedding(
            driver_id
        )

        x_race = self.race_embedding(
            race_id
        )

        x_compound = self.compound_embedding(
            compound_id
        )

        x_team = self.team_embedding(
            team_id
        )

        # ----------------------------------------------------
        # concatenate
        # ----------------------------------------------------

        x = torch.cat([
            x_tel,
            x_ctx,
            x_driver,
            x_race,
            x_compound,
            x_team
        ], dim=1)

        return self.head(x).squeeze()

In [140]:
gru_model = LapNet(
    telemetry_features=F,
    context_features=X_ctx.shape[1],
    num_drivers=num_drivers,
    num_races=num_races,
    num_compounds=num_compounds,
    num_teams=num_teams
)


criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    gru_model.parameters(),
    lr=1e-3
)

In [141]:
# ============================================================
# Training loop
# ============================================================

epochs = 10

for epoch in range(epochs):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    gru_model.train()

    train_loss = 0

    for (
        X_tel_batch,
        X_ctx_batch,
        driver_batch,
        race_batch,
        compound_batch,
        team_batch,
        y_batch
    ) in train_loader:

        optimizer.zero_grad()

        pred = gru_model(
            X_tel_batch,
            X_ctx_batch,
            driver_batch,
            race_batch,
            compound_batch,
            team_batch
        )

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(
        train_loader
    )

    # --------------------------------------------------------
    # TEST
    # --------------------------------------------------------

    gru_model.eval()

    test_loss = 0

    with torch.no_grad():

        for (
            X_tel_batch,
            X_ctx_batch,
            driver_batch,
            race_batch,
            compound_batch,
            team_batch,
            y_batch
        ) in test_loader:

            pred = gru_model(
                X_tel_batch,
                X_ctx_batch,
                driver_batch,
                race_batch,
                compound_batch,
                team_batch
            )

            loss = criterion(
                pred,
                y_batch
            )

            test_loss += loss.item()

    test_loss /= len(
        test_loader
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train: {train_loss:.4f} | "
        f"Test: {test_loss:.4f}"
    )

Epoch 01 | Train: 0.5076 | Test: 0.1799
Epoch 02 | Train: 0.2069 | Test: 0.1506
Epoch 03 | Train: 0.1780 | Test: 0.1594
Epoch 04 | Train: 0.1508 | Test: 0.1589
Epoch 05 | Train: 0.1269 | Test: 0.1202
Epoch 06 | Train: 0.1109 | Test: 0.1288
Epoch 07 | Train: 0.1059 | Test: 0.1266
Epoch 08 | Train: 0.1021 | Test: 0.1116
Epoch 09 | Train: 0.0983 | Test: 0.1111
Epoch 10 | Train: 0.0982 | Test: 0.1156


In [143]:
# ============================================================
# Evaluate GRU model
# ============================================================

gru_model.eval()

predictions = []
targets = []

with torch.no_grad():

    for (
        X_tel_batch,
        X_ctx_batch,
        driver_batch,
        race_batch,
        compound_batch,
        team_batch,
        y_batch
    ) in test_loader:

        pred = gru_model(
            X_tel_batch,
            X_ctx_batch,
            driver_batch,
            race_batch,
            compound_batch,
            team_batch
        )

        predictions.append(
            pred.numpy()
        )

        targets.append(
            y_batch.numpy()
        )

# ============================================================
# Concatenate
# ============================================================

predictions = np.concatenate(
    predictions
)

targets = np.concatenate(
    targets
)

# ============================================================
# Back to seconds
# ============================================================

pred_seconds = (
    predictions * y_std
    + y_mean
)

target_seconds = (
    targets * y_std
    + y_mean
)

# ============================================================
# Metrics
# ============================================================

mae = np.mean(
    np.abs(
        pred_seconds
        - target_seconds
    )
)

rmse = np.sqrt(
    np.mean(
        (
            pred_seconds
            - target_seconds
        ) ** 2
    )
)

print(f"MAE  (seconds): {mae:.3f}")
print(f"RMSE (seconds): {rmse:.3f}")

# ============================================================
# Show examples
# ============================================================

results = pd.DataFrame({

    "actual": target_seconds,
    "predicted": pred_seconds
})

results["error"] = (
    results["predicted"]
    - results["actual"]
)

display(results.head(20))

MAE  (seconds): 2.074
RMSE (seconds): 3.577


,actual,predicted,error
0,89.563004,89.450668,-0.112335
1,88.946999,89.570892,0.623894
2,89.290001,89.685242,0.395241
3,89.526001,89.825180,0.299179
4,89.419998,89.142624,-0.277374
5,89.392998,89.415222,0.022224
6,88.626999,89.503380,0.876381
7,88.751999,89.846985,1.094986
8,88.785004,88.982346,0.197342
9,89.060997,89.613518,0.552521


In [144]:
# ============================================================
# Classical ML baseline
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

# ============================================================
# Flatten telemetry
# ============================================================

N, T, F = X_tel_scaled.shape

X_tel_flat = X_tel_scaled.reshape(
    N,
    T * F
)

# ============================================================
# Combine all features
# ============================================================

X_all = np.concatenate([

    X_tel_flat,
    X_ctx_scaled,

    driver_ids.reshape(-1, 1),
    race_ids.reshape(-1, 1),
    compound_ids.reshape(-1, 1),
    team_ids.reshape(-1, 1)

], axis=1)

# ============================================================
# Train/test split
# ============================================================

X_train = X_all[train_idx]
X_test = X_all[test_idx]

y_train_rf = y[train_idx]
y_test_rf = y[test_idx]

# ============================================================
# Random Forest
# ============================================================

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)

rf.fit(
    X_train,
    y_train_rf
)

# ============================================================
# Predict
# ============================================================

pred = rf.predict(
    X_test
)

# ============================================================
# Metrics
# ============================================================

mae = mean_absolute_error(
    y_test_rf,
    pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test_rf,
        pred
    )
)

print(f"RF MAE:  {mae:.3f} sec")
print(f"RF RMSE: {rmse:.3f} sec")

RF MAE:  1.547 sec
RF RMSE: 3.390 sec
